In [1]:
from debugbox import DOSBoxDebugger
from pathlib import Path
import time
from collections import Counter
import subprocess

In [2]:
appname='/opt/homebrew/bin/dosbox-x'

def activate(app):
    subprocess.run(["osascript", "-e", f'tell application "System Events" to set frontmost of process "{app}" to true'])
def send_key_code(code, app=appname, modifier=None):
    subprocess.run(["osascript", "-e", f'tell application "{appname}" to activate'])
    mod = f" using {{{modifier}}}" if modifier else ""
    script = f'''
      tell application "System Events"
      tell process "{app}"
          key code {code}{mod}
      end tell
      end tell'''
    subprocess.run(["osascript", "-e", script])

In [3]:
import time

import re

ANSI_RE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
EV_RE = re.compile(r"LOG: EV of '([^']*)' is:\s*\n(?:LOG: )?([0-9a-fA-F]+(?:\s+[0-9a-fA-F]+)*)")

def get_registers(reg_names=("BX", "CS", "IP")):
    """
    Sends an EV command for the requested registers and reads the
    result back from the same logfile DOS MCBS writes to. Returns
    {reg_name: int_value}.
    """
    global last_pos
    new_text = dbg.ev(" ".join(reg_names))
    cleaned = ANSI_RE.sub('', new_text)
    m = EV_RE.search(cleaned)
    if not m:
        print('missing')
        return None
    names = m.group(1).split()
    values = [int(v, 16) for v in m.group(2).split()]
    ev_data = dict(zip(names, values))
    return ev_data


def run_until_break(reg_names=("BX", "CS", "IP"), max_wait=5.0, poll_interval=0.03):
    dbg.run()
    start = time.time()
    while time.time() - start < max_wait:
        regs = get_registers(reg_names)
        if regs is not None:
            return regs
        time.sleep(poll_interval)
    return None  # timed out -- worth logging/counting these separately

In [6]:
dbg = DOSBoxDebugger(
    conf_path="/Users/robmaxwell/Library/Preferences/f19-dosbox.conf",
    dump_path="/Users/robmaxwell/projects/f19/MEMDUMP.BIN",
)
dbg.start()

AUTOTYPE ENTER
LOG: Early LOG Init complete
LOG: CONFIG: Loaded config file: /Users/robmaxwell/Library/Preferences/f19-dosbox.conf
LOG: DOSBox-X's working directory: /Users/robmaxwell/projects/f19
LOG: Logging init: beginning logging proper. This is the end of the early init logging
LOG: Logging: opened logfile '/Users/robmaxwell/projects/f19/logs/f19-dosbox.log' successfully. All further logging will go to this file.
LOG: DOSBox-X version 2026.08.02 Commit e522642 (macOS ARM SDL2 64-bit)
LOG: macOS EXE path: /opt/homebrew/bin/dosbox-x
LOG: macOS Resource path: 
LOG: macOS: NSTouchBar support detected in system
LOG: Prevent capture: 0
LOG: SDL: version 2.32.70, Video cocoa, Audio coreaudio
LOG: Host keyboard layout is now us (US English)
LOG: Mapper keyboard layout is now us (US English)
LOG: SDL2 reports desktop display mode 1470 x 956
LOG: The default output for the video system: opengl
LOG: OpenGL Version : 2.1 Metal - 89.4
LOG: Configured windowposition: -
LOG: SDL: Current window 

In [25]:
#dbg.send("BP 1CE8:00001413")
dbg.send("BP 1CE8:00000E80")
#dbg.send("BP 1CE8:00001574")

dbg.run()
send_key_code(45, app=appname)
time.sleep(1)
send_key_code(18, app=appname)
for i in range(5):
    send_key_code(36, app=appname)
    time.sleep(1)

OSError: [Errno 5] Input/output error

In [11]:
z = dbg.command('DOS MCBS')





LOG: MCB Seg  Size (bytes)  PSP Seg (notes)  Filename
LOG: Conventional memory:
LOG:    0450         10992     0008 (DOS)    
LOG:    0700           720     072F          COMMAND
LOG:    072E          2464     072F          COMMAND
LOG:    07C9          1168     0814          F19
LOG:    0813          1840     0814          F19
LOG:    0887          5120     0814          F19
LOG:    09C8          2384     0814          F19
LOG:    0A5E           288     0814          F19
LOG:    0A71         10144     0814          F19
LOG:    0CEC         64000     0814          F19
LOG:    1C8D          1168     1CD8          START
LOG:    1CD7        131328     1CD8          START
LOG:    3CE8          6704     1CD8          START
LOG:    3E8C         12000     1CD8          START
LOG:    417B         64000     1CD8          START
LOG:    511C         64000     1CD8          START
LOG:    60BD         64000     1CD8          START
LOG:    705E         64000     1CD8          START
LOG:    7FFF

In [22]:
def parse_mcb(raw):
    
    clean = ANSI_RE.sub('', raw)
    mcb = {}
    
    in_table = False
    
    for line in clean.splitlines():
        line = line.strip()
    
        if line.startswith("LOG: MCB Seg"):
            in_table = True
            continue
    
        if not in_table:
            continue
    
        # Stop when we hit the next section
        if line == "LOG: Upper memory:":
            break
    
        # Remove the LOG: prefix
        if line.startswith("LOG:"):
            line = line[4:].strip()
    
        # Ignore blank lines and END OF CHAIN
        if not line or "END OF CHAIN" in line:
            continue
    
        parts = line.split()
    
        # Need at least:
        # segment, size, PSP
        if len(parts) < 3:
            continue
    
        segment = parts[0]
        size = int(parts[1])
        psp = parts[2]
    
        # Remaining text is notes / filename
        rest = parts[3:]
    
        notes = None
        filename = None
    
        if rest:
            # "(DOS)" / "(free)" etc.
            if rest[0].startswith("("):
                notes = rest[0]
                rest = rest[1:]
    
            if rest:
                filename = rest[0]
    
        mcb[segment] = {
            "size": size,
            "psp": psp,
            "notes": notes,
            "filename": filename,
        }
    return mcb

def find_owner(hex_addr, mcb):
    addr = int(hex_addr, 16)

    for segment, block in mcb.items():
        start = int(segment, 16)
        end = start + (block["size"] // 16)

        if start <= addr < end:
            return {
                "segment": segment,
                "filename": block["filename"],
                "size": block["size"],
                "start": start,
                "end": end,
            }

    return None

In [24]:
find_owner("1CE8", zz)

{'segment': '1CD7',
 'filename': 'START',
 'size': 131328,
 'start': 7383,
 'end': 15591}

In [137]:
dbg.send("MEMDUMPBIN")





LOG: DEBUG: Memory dump binary success.

_          

In [94]:
dbg.send("BPDEL *")
dbg.send("BP 1CE8:00001413")
dbg.run()





LOG: DEBUG: Breakpoints deleted.

      



LOG: DEBUG: Set breakpoint at 1CE8:1413

               


7mN_
(Running)                                                                      

1CE8:00003D7F 3946F0              cmp  [bp-10],ax             ss:[DA30]=0053    1CE8:00003D82 722C                jc   00003DB0 ($+2c)        (no jmp)          00003D826
96000000


1CE8:00001413 E9A80A              jmp  00001EBE ($+aa8)       (down)            1CE8:00001416 803E587E01          cmp  byte [7E58],01         ds:[7E58]=0000    141B 7510ne  0000142D ($+10)      141D 803E1A0A0F0A1A],0F   ds:[0A1A]=03311422 7609be142D ($+9) down)  1424 C646F400  movC],00  A62]=0000   
I-> _                                                                          027208866B400001413  C1 Z0 S1 O0 A1006EF0FA4C34240150
5235.323666667


In [95]:
dbg.run()




7mN_
(Running)                                                                      

1CE8:00001413 E9A80A              jmp  00001EBE ($+aa8)       (down)            00001EBE1
24333333


1CE8:00001413 E9A80A              jmp  00001EBE ($+aa8)       (down)            009

   
I-> _                                                                          687E0000014130017268452828339.626000000


In [96]:
%%capture cap --no-stderr

dbg.send("BPDEL *")
dbg.send("RUNWATCH")
time.sleep(3)
send_key_code(125, app=appname)
time.sleep(1)
send_key_code(36, app=appname)
time.sleep(1)
print("enter pressed!")
time.sleep(5)



1CE8:00001EB8 E83643call 000061F1 ($+4336)
1CE8:00001EBB E91EF5mp13DC ($-ae2)up)    
1CE8:00001EBE 9ABC0CE82Ccall 2CE8:0CBC
1CE8:00001EC3 0AC0or   al,al
1CE8:00001EC5 750Bjne  00001ED2 ($+b)(no jmp)
1CE8:00001EC7 9AC10CE82Ccall 2CE8:0CC1FF0000026A
1CE81EB8  C0 Z1 S0 O0 A0 P1

00000002C656700
45
3E5 837EF800mp  word [bp-08],0000ss:[DA56]=00003E9 7522  ne40D ($+22) down)3EB 837EF600  mp  word [bp-0A],0000ss:[DA54]=00003EF 751Cjne  0000140D ($+1c)(down)3F1 803EAA164Ecmp  byte [16AA],4Eds:[16AA]=80803F6 7215      jc   0000140D ($+15)(no jmp)3E50 S1757664
79


1CE8:000013E5 837EF800            cmp  word [bp-08],0000      ss:[DA56]=0000    no jmp)no jmp)
1CE8:000013F1 803EAA164E          cmp  byte [16AA],4E         ds:[16AA]=8080    F11 S08616269014


0A66:00000013 2BC0                sub  ax,ax                                    0A66:00000015 F7D0not  ax                     
0A66:00000017 CB      retf                                
0A66:00000018 06  push es                     
0A66:0000

In [97]:
with open('output_file2.txt', 'w') as f:
    f.write(cap.stdout)

In [98]:
def parse_mcb_chain(raw, first_mcb_segment):
    """
    raw: bytes of the full MCB-chain region, read starting at first_mcb_segment:0000
    first_mcb_segment: the segment of the first MCB in the chain
    """
    entries = []
    seg = first_mcb_segment
    pos = 0
    while True:
        record = raw[pos:pos+16]
        sig = chr(record[0])
        owner_psp = int.from_bytes(record[1:3], "little")
        size_paras = int.from_bytes(record[3:5], "little")
        name = record[8:16].split(b"\x00")[0].strip().decode("ascii", "replace")

        entries.append({
            "mcb_segment": seg,
            "owned_start": seg + 1,
            "owned_end": seg + size_paras,   # inclusive-ish; block spans seg+1 .. seg+size_paras
            "owner_psp": owner_psp,
            "size_paras": size_paras,
            "name": name,
        })

        if sig == "Z":
            break

        next_seg = seg + 1 + size_paras
        pos += (next_seg - seg) * 16   # advance raw buffer by the same distance, in bytes
        seg = next_seg

    return entries

def find_owner(entries, target_segment):
    for e in entries:
        if e["owned_start"] <= target_segment <= e["owned_end"]:
            return e
    return None

In [130]:
with open('MEMDUMP.BIN', 'rb') as f:
    raw = f.read()

In [119]:
raw[guess_offset]

75

In [124]:
0xF000

61440

In [136]:
chain = try_walk(raw, 0x4510)

for offset, sig, owner_psp, size_paras, name in chain:
    seg = offset // 16
    owned_start = seg + 1
    owned_end = seg + size_paras
    hit = " <=== HIT" if owned_start <= 0x1CE8 <= owned_end else ""
    print(f"mcb_seg={seg:04X}  owner_psp={owner_psp:04X}  "
          f"owns {owned_start:04X}-{owned_end:04X}  size={size_paras:04X}  name={name!r}{hit}")

mcb_seg=0451  owner_psp=0000  owns 0452-9FFE  size=9BAD  name='' <=== HIT


In [122]:
from collections import Counter
segs = [int.from_bytes(raw[i+2:i+4], 'little') for i in range(0, 0x400, 4)]
print(Counter(segs).most_common(5))

[(0, 143), (61440, 94), (112, 5), (7400, 3), (49152, 3)]


In [131]:
guess_offset = 0x4510
print(hex(guess_offset), chr(raw[guess_offset]) if raw[guess_offset] in (0x4D, 0x5A) else raw[guess_offset])

0x4510 Z


In [132]:
def try_walk(raw, start):
    offset = start
    chain = []
    seen = set()
    while True:
        if offset < 0 or offset + 16 > len(raw) or offset in seen:
            return None
        seen.add(offset)

        sig = raw[offset]
        if sig not in (0x4D, 0x5A):          # must be 'M' or 'Z'
            return None

        owner_psp = int.from_bytes(raw[offset+1:offset+3], 'little')
        size_paras = int.from_bytes(raw[offset+3:offset+5], 'little')
        name = raw[offset+8:offset+16].split(b'\x00')[0].decode('ascii', 'replace')
        chain.append((offset, chr(sig), owner_psp, size_paras, name))

        if sig == 0x5A:            # 'Z' = end of chain, and we made it here cleanly
            return chain
        if size_paras == 0:
            return None

        offset += (size_paras + 1) * 16   # this MCB + the block it owns

best, best_start = None, None
for start in range(0, len(raw) - 16, 16):   # every paragraph boundary
    chain = try_walk(raw, start)
    if chain and (best is None or len(chain) > len(best)):
        best, best_start = chain, start

if best:
    print(f"chain starts at file offset {hex(best_start)}, {len(best)} entries, ends in {best[-1][1]!r}")
    for entry in best:
        print(entry)
else:
    print("nothing chained cleanly — see notes below")

chain starts at file offset 0x4510, 1 entries, ends in 'Z'
(17680, 'Z', 0, 39853, '')


In [ ]:
1CE8:00003D7F

In [133]:
import re

name_pattern = re.compile(rb'[A-Za-z0-9_]{1,8}\.(?:EXE|COM)\x00*', re.IGNORECASE)

records = []
for m in name_pattern.finditer(raw):
    name_off = m.start()
    record_start = name_off - 8
    if record_start < 0:
        continue
    sig = raw[record_start]
    # if sig not in (0x4D, 0x5A):   # 'M' or 'Z' — real MCB header, not a coincidental substring
    #     continue
    owner_psp = int.from_bytes(raw[record_start+1:record_start+3], 'little')
    size_paras = int.from_bytes(raw[record_start+3:record_start+5], 'little')
    name = m.group().split(b'\x00')[0].decode('ascii', 'replace')
    records.append((record_start, sig, owner_psp, size_paras, name))
    print(f"file_offset={record_start:#07x}  sig={chr(sig)}  owner_psp={owner_psp:#06x}  "
          f"size_paras={size_paras:#06x}  name={name!r}")

file_offset=0x0705d  sig=S  owner_psp=0x4550  size_paras=0x3d43  name='COMMAND.COM'
file_offset=0x07aa6  sig=   owner_psp=0x0000  size_paras=0x0000  name='F19.COM'
file_offset=0x07ab0  sig=9  owner_psp=0x432e  size_paras=0x4d4f  name='mount.COM'
file_offset=0x07ced  sig=S  owner_psp=0x4550  size_paras=0x3d43  name='COMMAND.COM'
file_offset=0x07d29  sig=6  owner_psp=0x0000  size_paras=0x0001  name='F19.COM'
file_offset=0x07d31  sig=F  owner_psp=0x3931  size_paras=0x432e  name='OUNT.COM'
file_offset=0x0861e  sig=:  owner_psp=0x635c  size_paras=0x625c  name='cv.exe'
file_offset=0x08625  sig=\  owner_psp=0x7663  size_paras=0x652e  name='mcv.exe'
file_offset=0x0862d  sig=m  owner_psp=0x7663  size_paras=0x652e  name='SU.EXE'
file_offset=0x08634  sig=   owner_psp=0x5553  size_paras=0x452e  name='START.EXE'
file_offset=0x0863e  sig=A  owner_psp=0x5452  size_paras=0x452e  name='missions.EXE'
file_offset=0x0864b  sig=o  owner_psp=0x736e  size_paras=0x452e  name='EGAME.EXE'
file_offset=0x08655  s

In [114]:
records

[]

In [134]:
# with open('MEMDUMP.BIN', 'r', encoding='utf-8', errors='replace') as f:
#     lines = f.readlines()

# import re
# ansi_escape = re.compile(r'\x1b\[[0-9;]*[a-zA-Z]')
# clean_lines = [ansi_escape.sub('', l) for l in lines]

# for line in clean_lines[:60]:
#     print(repr(line))

In [135]:
# #1CE8:00003D7F
# parse_mcb_chain(text, )

In [160]:
import json
with open("f19_trace_data2.json", "r") as file:
    data = json.load(file)

In [166]:
candidates = list(data['22']['pairs'].keys())

In [177]:
possible = []
for c in candidates:
    for i in range(22):
        if c in list(data[str(i)]['pairs'].keys()):
            continue
        else:
            possible.append(c)

In [178]:
len(set(possible))

2854

In [174]:
len(set(candidates))

2883

In [176]:
c in data[str(i)]['pairs'].keys()

False

In [191]:
import numpy as np
{k:v for k, v in data['22']['pairs'].items() if v>10 and k in possible}

{'1CE8:000083E1': 97,
 '1CE8:000083E6': 97,
 '1CE8:000083CE': 97,
 '1CE8:000083D1': 103,
 '1CE8:000083D5': 103,
 '1CE8:000083D7': 96,
 '1CE8:000083DA': 96,
 '1CE8:000083DC': 96,
 '1CE8:000083DE': 96,
 '1CE8:0000E372': 20,
 '1CE8:0000E373': 20,
 '1CE8:0000E375': 20,
 '1CE8:0000E378': 20,
 '1CE8:0000E37B': 20,
 '1CE8:0000E37D': 20,
 '1CE8:0000E380': 20,
 '1CE8:0000E458': 150,
 '1CE8:0000E45A': 150,
 '1CE8:0000E45C': 150,
 '1CE8:00005124': 80,
 '1CE8:00005127': 80,
 '1CE8:0000512A': 80,
 '1CE8:0000512D': 80,
 '1CE8:00005130': 80,
 '1CE8:00005132': 80,
 '1CE8:00005134': 80,
 '1CE8:00005136': 80,
 '1CE8:0000B671': 17,
 '1CE8:0000B675': 17,
 '1CE8:0000B677': 16,
 '1CE8:0000B67C': 16,
 '1CE8:0000B650': 272,
 '1CE8:0000B654': 272,
 '1CE8:0000B656': 256,
 '1CE8:0000B659': 256,
 '1CE8:0000B65B': 256,
 '1CE8:0000B65D': 256,
 '1CE8:0000B660': 256,
 '1CE8:0000B665': 256,
 '1CE8:0000B64D': 256,
 '1CE8:0000B66E': 16,
 '1CE8:0000B690': 48,
 '1CE8:0000B693': 48,
 '1CE8:0000B696': 48,
 '1CE8:0000B685': 